<div style="font-size: 13px; line-height: 1.25;">

# UN Comtrade Services Trade Data Pull — EBOPS (As Reported)

This notebook pulls **bilateral services trade data (imports)** from the **UN Comtrade v1 API** for **all individual countries**, **all available EBOPS service codes**, and **years 2000–2025**. Results are combined into a **single deduplicated CSV**.

**Setup:** Jupyter Notebook, **R kernel**  
**Classification:** **EBOPS (As Reported)**  
**Flow:** **Imports only**

</div>

<div style="font-size: 25px; line-height: 1.25;">
SECTION 0 — SETUP
</div>

In [ ]:
# --- 0.1 Install / load required packages ------------------------------------
 
required_packages <- c("httr", "jsonlite", "dplyr", "purrr")
 
for (pkg in required_packages) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
  }
  library(pkg, character.only = TRUE)
}
 
cat("All packages loaded.\n")
 
# --- 0.2 User configuration block --------------------------------------------
# *** EDIT THESE VALUES BEFORE RUNNING ***
 
# Your Comtrade API subscription key (obtained from https://comtradedeveloper.un.org)
SUBSCRIPTION_KEY_1 <- Sys.getenv("COMTRADE_KEY_1")  # set in your environment; never commit keys
SUBSCRIPTION_KEY_2 <- Sys.getenv("COMTRADE_KEY_2")
# Output file path for the final CSV
OUTPUT_FILE <- "../data/raw/data_result.csv"
 
# --- 0.3 Fixed API parameters ------------------------------------------------
# These match the specifications in the prompt. Change only if you know what
# you are doing.
 
TYPE_CODE   <- "S"       # S = Services (vs C = Commodities)
FREQ_CODE   <- "A"       # A = Annual
CL_CODE     <- "EB"      # EB = EBOPS As Reported
FLOW_CODE   <- "M"       # M = Imports
BREAKDOWN   <- "plus"    # Breakdown mode: plus (extended)
AGGREGATE   <- "none"    # No aggregation
INCLUDE_DESC <- "true"   # Include text descriptions in the response
MAX_RECORDS <- 250000    # API hard ceiling per call
 
# Year range
YEAR_START <- 2000
YEAR_END   <- 2025
 
# Batch size (API maximum is 12 years per call)
YEAR_BATCH_SIZE <- 12
 
# Rate-limiting / retry settings
RATE_LIMIT_PAUSE <- 6    # Seconds to wait between API calls (respects ~10/min)
MAX_RETRIES      <- 5    # Maximum retry attempts per failed request
BACKOFF_BASE     <- 10   # Base wait (seconds) for exponential backoff
 
# --- 0.4 API endpoint constants ----------------------------------------------
 
BASE_DATA_URL <- "https://comtradeapi.un.org/data/v1/get"
BASE_REF_URL  <- "https://comtradeapi.un.org/files/v1/app/reference"
 
# Full data endpoint for services / annual / EBOPS-as-reported
DATA_ENDPOINT <- paste(BASE_DATA_URL, TYPE_CODE, FREQ_CODE, CL_CODE, sep = "/")
 
cat("Data endpoint:", DATA_ENDPOINT, "\n")


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'purrr'


The following object is masked from 'package:jsonlite':

    flatten




All packages loaded.
Data endpoint: https://comtradeapi.un.org/data/v1/get/S/A/EB 


<div style="font-size: 25px; line-height: 1.25;">
SECTION 1 — FETCH REFERENCE LISTS
</div>

In [4]:
# Pull reporters, partners, and service codes from the Comtrade reference API,
# then filter to individual countries only (excluding aggregates).
 
# --- 1.1 Helper: safe GET for reference JSON ---------------------------------
 
fetch_reference <- function(endpoint_url) {
  # Fetches a Comtrade reference JSON and returns it as a data.frame.
  resp <- tryCatch(
    httr::GET(endpoint_url, httr::timeout(60)),
    error = function(e) {
      warning("HTTP error fetching reference: ", e$message)
      return(NULL)
    }
  )
  if (is.null(resp) || httr::status_code(resp) != 200) {
    stop("Failed to fetch reference from: ", endpoint_url,
         " (status ", httr::status_code(resp), ")")
  }
  raw <- httr::content(resp, as = "text", encoding = "UTF-8")
  parsed <- jsonlite::fromJSON(raw, flatten = TRUE)
  # Reference JSONs wrap data inside a "results" key
  if ("results" %in% names(parsed)) {
    return(as.data.frame(parsed$results))
  } else {
    return(as.data.frame(parsed))
  }
}
 
# --- 1.2 Reporters -----------------------------------------------------------
 
cat("\n--- Fetching reporter list ---\n")
reporters_url <- paste0(BASE_REF_URL, "/Reporters.json")
reporters_raw <- fetch_reference(reporters_url)
 
# The reference JSON contains fields like:
#   id            — numeric country/area code
#   text          — country name
#   reporterNote  — additional info (may be absent)
#   isGroup       — TRUE for aggregates / groupings
#
# EXCLUSION LOGIC:
# 1. Drop any entry where isGroup == TRUE (regional/economic aggregates).
# 2. Drop id == 0 ("World").
# 3. Drop entries whose text matches known aggregate patterns:
#    "World", "All", "ASEAN", "EU-", "EFTA", "OECD", "OPEC",
#    "Least developed", "Developing", "Developed", "income",
#    "Africa", "Asia", "Europe", "Americas", "Oceania",
#    "Small island", "Landlocked", "Unspecified", "Other Asia",
#    "Special Categories", "Neutral Zone", "Free Zones", "Bunkers".
# This two-layer approach (isGroup flag + name pattern) ensures robustness
# even if the isGroup flag is inconsistent for some entries.
 
aggregate_patterns <- paste(
  c("World", "^All ", "ASEAN", "EU-", "EU28", "EFTA", "OECD", "OPEC",
    "Least developed", "Developing", "Developed",
    "income", "Africa", "Asia", "Europe", "Americas", "Oceania",
    "Small island", "Landlocked", "Unspecified", "Other Asia, nes",
    "Special Categories", "Neutral Zone", "Free Zones", "Bunkers",
    "Areas, nes", "Other Europe, nes", "Other Africa, nes",
    "Western Hemisphere, nes", "Rest of"),
  collapse = "|"
)
 
reporters <- reporters_raw %>%
  filter(
    # Keep only non-group entries
    (is.na(isGroup) | isGroup == FALSE),
    # Exclude World (id 0)
    id != 0,
    # Exclude names matching aggregate patterns
    !grepl(aggregate_patterns, text, ignore.case = TRUE)
  ) %>%
  select(id, text) %>%
  arrange(id)
 
cat("Reporters retrieved:", nrow(reporters), "individual countries\n")
 
# --- 1.3 Partners ------------------------------------------------------------
 
cat("\n--- Fetching partner list ---\n")
partners_url <- paste0(BASE_REF_URL, "/partnerAreas.json")
partners_raw <- fetch_reference(partners_url)
 
# Apply the same exclusion logic as reporters.
partners <- partners_raw %>%
  filter(
    (is.na(isGroup) | isGroup == FALSE),
    id != 0,
    !grepl(aggregate_patterns, text, ignore.case = TRUE)
  ) %>%
  select(id, text) %>%
  arrange(id)
 
cat("Partners retrieved:", nrow(partners), "individual countries\n")
 
# --- 1.4 EBOPS service codes -------------------------------------------------
 
cat("\n--- Fetching EBOPS service code list ---\n")
# The classification code for "EBOPS as reported" is "EB".
# Reference file follows the pattern: <classification>.json
services_url <- paste0(BASE_REF_URL, "/", CL_CODE, ".json")
services_raw <- fetch_reference(services_url)
 
# Keep all available codes — no filtering needed here.
service_codes <- services_raw %>%
  select(id, text) %>%
  arrange(id)
 
cat("Service codes retrieved:", nrow(service_codes), "\n")
 
# --- 1.5 Summary print -------------------------------------------------------
 
cat("\n╔══════════════════════════════════════╗\n")
cat("║   REFERENCE DATA SUMMARY            ║\n")
cat("╠══════════════════════════════════════╣\n")
cat(sprintf("║  Reporters    : %4d countries       ║\n", nrow(reporters)))
cat(sprintf("║  Partners     : %4d countries       ║\n", nrow(partners)))
cat(sprintf("║  Service codes: %4d EBOPS codes     ║\n", nrow(service_codes)))
cat("╚══════════════════════════════════════╝\n")


--- Fetching reporter list ---
Reporters retrieved: 249 individual countries

--- Fetching partner list ---
Partners retrieved: 290 individual countries

--- Fetching EBOPS service code list ---
Service codes retrieved: 423 

╔══════════════════════════════════════╗
║   REFERENCE DATA SUMMARY            ║
╠══════════════════════════════════════╣
║  Reporters    :  249 countries       ║
║  Partners     :  290 countries       ║
║  Service codes:  423 EBOPS codes     ║
╚══════════════════════════════════════╝


In [7]:
print(as.data.frame(service_codes))

            id
1            1
2          1.1
3          1.2
4           10
5         10.1
6       10.1.1
7     10.1.1.1
8     10.1.1.2
9   10.1.1.2.1
10  10.1.1.2.2
11  10.1.1.2.3
12  10.1.1.2.4
13      10.1.2
14        10.2
15      10.2.1
16    10.2.1.1
17    10.2.1.2
18    10.2.1.3
19      10.2.2
20    10.2.2.1
21        10.3
22      10.3.1
23    10.3.1.1
24    10.3.1.2
25    10.3.1.3
26      10.3.2
27    10.3.2.1
28    10.3.2.2
29    10.3.2.3
30      10.3.3
31      10.3.4
32      10.3.5
33    10.3.5.1
34          11
35        11.1
36      11.1.1
37    11.1.1.a
38      11.1.2
39        11.2
40      11.2.1
41      11.2.2
42      11.2.3
43      11.2.4
44          12
45        12.1
46        12.2
47        12.3
48           2
49         200
50         205
51         206
52         207
53         208
54         209
55         210
56         211
57         212
58         213
59         214
60         215
61         216
62         217
63         218
64         219
65         220
66        

<div style="font-size: 25px; line-height: 1.25;">
SECTION 2 — BATCHING AND API CALLS
</div>

In [4]:
# Force immediate console output in Jupyter
options(jupyter.rich_display = FALSE)
 
# --- 2.1 Build year batches ---------------------------------------------------
 
all_years   <- YEAR_START:YEAR_END
year_chunks <- split(all_years, ceiling(seq_along(all_years) / YEAR_BATCH_SIZE))
 
cat("\nYear batches created:", length(year_chunks), "\n")
for (i in seq_along(year_chunks)) {
  cat(sprintf("  Batch %d: %d–%d (%d years)\n",
              i, min(year_chunks[[i]]), max(year_chunks[[i]]),
              length(year_chunks[[i]])))
}
 
# --- 2.2 Build reporter batches -----------------------------------------------
 
REPORTER_BATCH_SIZE <- 5
 
reporter_ids    <- as.character(reporters$id)
reporter_chunks <- split(reporter_ids,
                         ceiling(seq_along(reporter_ids) / REPORTER_BATCH_SIZE))
 
cat("Reporter batches created:", length(reporter_chunks),
    "(", REPORTER_BATCH_SIZE, "reporters each)\n")
 
# --- 2.3 Build partner batches ------------------------------------------------
# To get bilateral data we must pass individual partner codes instead of "0".
# We batch partners in groups of 50 to keep URLs manageable and response sizes
# within the 250K row limit.
 
PARTNER_BATCH_SIZE <- 50
 
partner_ids    <- as.character(partners$id)
partner_chunks <- split(partner_ids,
                        ceiling(seq_along(partner_ids) / PARTNER_BATCH_SIZE))
 
cat("Partner batches created:", length(partner_chunks),
    "(", PARTNER_BATCH_SIZE, "partners each)\n")
 
# --- 2.4 Dual-key rotation setup ---------------------------------------------
# Two subscription keys give us 500 + 500 = 1000 calls per day.
# We alternate between them to spread usage evenly.
 
api_keys     <- c(SUBSCRIPTION_KEY_1, SUBSCRIPTION_KEY_2)
call_counter <- 0  # tracks total calls to rotate keys
 
get_current_key <- function() {
  # Alternates between the two keys each call
  api_keys[((call_counter) %% 2) + 1]
}
 
# --- 2.5 Reusable API call function with retry logic -------------------------
 
pull_comtrade_batch <- function(year_vec, reporter_vec, partner_vec,
                                attempt = 1, max_attempts = MAX_RETRIES) {
  # Constructs a Comtrade v1 API URL and returns the parsed data.frame.
  #
  # Args:
  #   year_vec     — integer vector of years (max 12)
  #   reporter_vec — character vector of reporter codes
  #   partner_vec  — character vector of partner codes
  #   attempt      — current retry attempt (used internally)
  #   max_attempts — maximum number of retries
  #
  # Returns:
  #   A data.frame of trade records, or NULL if all retries failed.
 
  period_str   <- paste(year_vec, collapse = ",")
  reporter_str <- paste(reporter_vec, collapse = ",")
  partner_str  <- paste(partner_vec, collapse = ",")
 
  # Construct query parameters explicitly
  query_params <- list(
    reporterCode       = reporter_str,
    period             = period_str,
    partnerCode        = partner_str,
    partner2Code       = "0",
    flowCode           = FLOW_CODE,
    breakdownMode      = BREAKDOWN,
    includeDesc        = INCLUDE_DESC,
    `subscription-key` = get_current_key()
  )
 
  # --- Make the request -------------------------------------------------------
  resp <- tryCatch(
    httr::GET(DATA_ENDPOINT, query = query_params, httr::timeout(120)),
    error = function(e) {
      warning("  HTTP request error: ", e$message)
      return(NULL)
    }
  )
 
  # --- Handle failures --------------------------------------------------------
  if (is.null(resp)) {
    if (attempt < max_attempts) {
      wait <- BACKOFF_BASE * 2^(attempt - 1)
      cat(sprintf("    Retry %d/%d in %d seconds...\n",
                  attempt, max_attempts, wait))
      flush.console()
      Sys.sleep(wait)
      return(pull_comtrade_batch(year_vec, reporter_vec, partner_vec,
                                 attempt + 1, max_attempts))
    } else {
      warning("  All retries exhausted — returning NULL.")
      return(NULL)
    }
  }
 
  status <- httr::status_code(resp)
 
  # Rate-limited (HTTP 429) or server error (5xx) → retry with backoff
  if (status == 429 || status >= 500) {
    if (attempt < max_attempts) {
      wait <- BACKOFF_BASE * 2^(attempt - 1)
      cat(sprintf("    HTTP %d — retry %d/%d in %d seconds...\n",
                  status, attempt, max_attempts, wait))
      flush.console()
      Sys.sleep(wait)
      return(pull_comtrade_batch(year_vec, reporter_vec, partner_vec,
                                 attempt + 1, max_attempts))
    } else {
      warning(sprintf("  HTTP %d after %d retries — giving up.", status, max_attempts))
      return(NULL)
    }
  }
 
  # Other client errors (4xx except 429) → do not retry
  if (status >= 400 && status < 500) {
    body_text <- httr::content(resp, as = "text", encoding = "UTF-8")
    warning(sprintf("  HTTP %d (client error). Body: %s",
                    status, substr(body_text, 1, 500)))
    return(NULL)
  }
 
  # --- Parse JSON response ----------------------------------------------------
  raw_text <- httr::content(resp, as = "text", encoding = "UTF-8")
 
  parsed <- tryCatch(
    jsonlite::fromJSON(raw_text, flatten = TRUE),
    error = function(e) {
      warning("  JSON parse error: ", e$message)
      return(NULL)
    }
  )
 
  if (is.null(parsed)) return(NULL)
 
  # The v1 API wraps data in a 'data' element
  if ("data" %in% names(parsed) && is.data.frame(parsed$data) && nrow(parsed$data) > 0) {
    return(parsed$data)
  } else {
    return(NULL)
  }
}
 
# --- 2.6 Main loop: iterate over reporter × partner × year batches ------------
 
total_batches <- length(reporter_chunks) * length(partner_chunks) * length(year_chunks)
batch_results <- vector("list", total_batches)
failed_batches <- list()
 
cat("\n╔══════════════════════════════════════════════════════════════╗\n")
cat("║   STARTING BILATERAL DATA PULL                             ║\n")
cat(sprintf("║   %d reporters × %d partners × %d year batches = %d calls\n",
            length(reporter_chunks), length(partner_chunks),
            length(year_chunks), total_batches))
cat(sprintf("║   Available daily quota: 1000 (2 keys × 500)             \n"))
cat("╚══════════════════════════════════════════════════════════════╝\n\n")
flush.console()
 
counter <- 0
pull_start <- Sys.time()
 
for (ri in seq_along(reporter_chunks)) {
  for (pi in seq_along(partner_chunks)) {
    for (yi in seq_along(year_chunks)) {
      counter      <- counter + 1
      call_counter <- call_counter + 1
 
      rpt_ids   <- reporter_chunks[[ri]]
      prt_ids   <- partner_chunks[[pi]]
      yr_range  <- year_chunks[[yi]]
 
      rpt_label <- paste(rpt_ids, collapse = ",")
      prt_label <- paste0("partner batch ", pi, "/", length(partner_chunks))
      yr_label  <- paste0(min(yr_range), "–", max(yr_range))
      key_label <- ifelse((call_counter %% 2) == 1, "Key1", "Key2")
 
      cat(sprintf("[%d/%d] Reporters {%s} | %s | Years %s [%s] ... ",
                  counter, total_batches, rpt_label, prt_label, yr_label, key_label))
      flush.console()
 
      result <- pull_comtrade_batch(yr_range, rpt_ids, prt_ids)
 
      if (is.null(result)) {
        cat("EMPTY / FAILED\n")
        flush.console()
        failed_batches[[length(failed_batches) + 1]] <- list(
          reporters = rpt_ids, partners = prt_ids, years = yr_range
        )
      } else {
        cat(sprintf("OK — %d rows\n", nrow(result)))
        flush.console()
        batch_results[[counter]] <- result
      }
 
      # Respect rate limit
      Sys.sleep(RATE_LIMIT_PAUSE)
    }
  }
}
 
pull_end <- Sys.time()
cat(sprintf("\nData pull completed in %.1f minutes.\n",
            as.numeric(difftime(pull_end, pull_start, units = "mins"))))
 
# --- 2.7 Report failed batches ------------------------------------------------
 
if (length(failed_batches) > 0) {
  cat("\n⚠ WARNING:", length(failed_batches),
      "batch(es) returned no data or failed after retries:\n")
  for (fb in failed_batches) {
    cat(sprintf("  Reporters: %s | Partners batch: %s | Years: %s\n",
                paste(fb$reporters, collapse = ","),
                paste(fb$partners[1:3], collapse = ","),
                paste(range(fb$years), collapse = "–")))
  }
  cat("These may be country pairs with no services trade data for those years.\n")
} else {
  cat("\nAll batches completed successfully.\n")
}
 


Year batches created: 3 
  Batch 1: 2000–2011 (12 years)
  Batch 2: 2012–2023 (12 years)
  Batch 3: 2024–2025 (2 years)
Reporter batches created: 50 ( 5 reporters each)
Partner batches created: 6 ( 50 partners each)

╔══════════════════════════════════════════════════════════════╗
║   STARTING BILATERAL DATA PULL                             ║
║   50 reporters × 6 partners × 3 year batches = 900 calls
║   Available daily quota: 1000 (2 keys × 500)             
╚══════════════════════════════════════════════════════════════╝

[1/900] Reporters {4,8,12,20,24} | partner batch 1/6 | Years 2000–2011 [Key1] ... EMPTY / FAILED
[2/900] Reporters {4,8,12,20,24} | partner batch 1/6 | Years 2012–2023 [Key2] ... OK — 1348 rows
[3/900] Reporters {4,8,12,20,24} | partner batch 1/6 | Years 2024–2025 [Key1] ... EMPTY / FAILED
[4/900] Reporters {4,8,12,20,24} | partner batch 2/6 | Years 2000–2011 [Key2] ... EMPTY / FAILED
[5/900] Reporters {4,8,12,20,24} | partner batch 2/6 | Years 2012–2023 [Key1] ...

<div style="font-size: 25px; line-height: 1.25;">
SECTION 3 — ASSEMBLE FINAL DATASET
</div>

In [5]:
# --- 3.1 Row-bind all batch results ------------------------------------------
 
cat("\n--- Assembling final dataset ---\n")
 
# Drop NULL entries (failed / empty batches)
batch_results_clean <- purrr::compact(batch_results)
 
if (length(batch_results_clean) == 0) {
  stop("No data was retrieved. Check your subscription keys and API access.")
}
 
# bind_rows handles differing column sets gracefully
full_df <- dplyr::bind_rows(batch_results_clean)
cat("Rows after binding:", format(nrow(full_df), big.mark = ","), "\n")
 
# --- 3.2 Deduplicate ---------------------------------------------------------
# Batching may introduce overlapping rows. Remove exact duplicates.
 
before_dedup <- nrow(full_df)
full_df <- dplyr::distinct(full_df)
after_dedup <- nrow(full_df)
 
cat(sprintf("Deduplication: %s → %s rows (%d duplicates removed)\n",
            format(before_dedup, big.mark = ","),
            format(after_dedup, big.mark = ","),
            before_dedup - after_dedup))
 
# --- 3.3 Validation summary --------------------------------------------------
 
cat("\n╔══════════════════════════════════════════════════════════════╗\n")
cat("║   FINAL DATASET VALIDATION                                 ║\n")
cat("╠══════════════════════════════════════════════════════════════╣\n")
cat(sprintf("║  Total rows           : %s\n", format(nrow(full_df), big.mark = ",")))
cat(sprintf("║  Total columns        : %d\n", ncol(full_df)))
 
if ("period" %in% names(full_df)) {
  yr_range_actual <- range(as.integer(full_df$period), na.rm = TRUE)
  cat(sprintf("║  Year range           : %d – %d\n", yr_range_actual[1], yr_range_actual[2]))
 
  # Check for gaps in year coverage
  years_present <- sort(unique(as.integer(full_df$period)))
  expected_years <- YEAR_START:YEAR_END
  missing_years <- setdiff(expected_years, years_present)
  if (length(missing_years) > 0) {
    cat(sprintf("║  ⚠ Missing years     : %s\n", paste(missing_years, collapse = ", ")))
  } else {
    cat("║  Year coverage        : Complete (no gaps)\n")
  }
}
 
if ("reporterCode" %in% names(full_df)) {
  cat(sprintf("║  Unique reporters     : %d\n", dplyr::n_distinct(full_df$reporterCode)))
}
if ("partnerCode" %in% names(full_df)) {
  cat(sprintf("║  Unique partners      : %d\n", dplyr::n_distinct(full_df$partnerCode)))
}
if ("cmdCode" %in% names(full_df)) {
  cat(sprintf("║  Unique service codes : %d\n", dplyr::n_distinct(full_df$cmdCode)))
}
 
cat("╚══════════════════════════════════════════════════════════════╝\n")
 


--- Assembling final dataset ---
Rows after binding: 1,486,955 
Deduplication: 1,486,955 → 1,486,955 rows (0 duplicates removed)

╔══════════════════════════════════════════════════════════════╗
║   FINAL DATASET VALIDATION                                 ║
╠══════════════════════════════════════════════════════════════╣
║  Total rows           : 1,486,955
║  Total columns        : 47
║  Year range           : 2000 – 2024
║  ⚠ Missing years     : 2025
║  Unique reporters     : 56
║  Unique partners      : 246
║  Unique service codes : 340
╚══════════════════════════════════════════════════════════════╝


<div style="font-size: 25px; line-height: 1.25;">
SECTION 4 — EXPORT
</div>

In [6]:
cat("\n--- Exporting to CSV ---\n")
 
write.csv(full_df, file = OUTPUT_FILE, row.names = FALSE, fileEncoding = "UTF-8")
 
file_size_mb <- round(file.info(OUTPUT_FILE)$size / 1e6, 1)
 
cat("\n╔══════════════════════════════════════════════════════════════╗\n")
cat("║   EXPORT COMPLETE                                          ║\n")
cat("╠══════════════════════════════════════════════════════════════╣\n")
cat(sprintf("║  File path : %s\n", OUTPUT_FILE))
cat(sprintf("║  Rows      : %s\n", format(nrow(full_df), big.mark = ",")))
cat(sprintf("║  Columns   : %d\n", ncol(full_df)))
cat(sprintf("║  File size : %.1f MB\n", file_size_mb))
cat("╚══════════════════════════════════════════════════════════════╝\n")
 
cat("\nDone. ✓\n")
 


--- Exporting to CSV ---

╔══════════════════════════════════════════════════════════════╗
║   EXPORT COMPLETE                                          ║
╠══════════════════════════════════════════════════════════════╣
║  File path : ../data/raw/data_result.csv
║  Rows      : 1,486,955
║  Columns   : 47
║  File size : 424.7 MB
╚══════════════════════════════════════════════════════════════╝

Done. ✓
